In [ ]:
!pip install tensorflowjs

In [ ]:
import tensorflow as tf

In [ ]:
def psnr_metric(y_true, y_pred):
    # Calculate PSNR manually to avoid Keras graph-mode Assert issues in tf.image.psnr
    # We add a small epsilon (1e-10) to the MSE to prevent the result from becoming 'inf'
    mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=[1, 2, 3])
    # PSNR = 10 * log10(max_val^2 / MSE). Here max_val is 1.0.
    # tf.math.log is natural log, so we divide by log(10) to get log10.
    return 10.0 * (tf.math.log(1.0 / (mse + 1e-10)) / tf.math.log(10.0))


In [ ]:
model = tf.keras.models.load_model(
    "./upscale_dummy.keras",
    custom_objects={
        "psnr_metric": psnr_metric
    }
)
model.summary()

In [ ]:
model.export("./tf")

In [ ]:
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_node_names='resizing' \
    --saved_model_tags=serve \
    ./tf \
    ./tfjs